# 2. Train

Trains the **lightweight direct mapping** (`MODEL = "direct"`) or, for the ablation, the **bridge-flow mapping** (`MODEL = "bridge_flow"`). Both use the same 9-block network, the same losses, data and budget:

| | direct | bridge_flow |
|---|---|---|
| Network | `DirectMapping`, 10.9M | velocity network: the same blocks + time conditioning, 12.4M |
| Trained at | t = 0 only (direct regression) | t ~ U[0, 1], bridge noise σ = 0.5, trajectory training on 50% of steps |
| Losses | latent MSE + VGG19 (10) + conditional multi-scale LSGAN (1) + feature matching (10), GAN from step 2,000 | same |
| Optimisation | AdamW 2e-4, cosine, weight decay 1e-4, batch 8, 20,000 steps, EMA 0.999 | same |

**Data.** A folder of clean photos with both sides ≥ 256 px. The paper uses 4,577 PASCAL VOC images. Pairs are degraded on the fly, with 50% grayscale; 5% of the images are held out for validation.

**Resumable.** `latest.pt` is written at every 500-step evaluation. After a disconnect, run all cells again.

**Keep at least 2 GB free on Google Drive.** Each checkpoint is about 265 MB, and on a full Drive, Colab's uploads fail without an error.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
if not os.path.isdir("/content/Bringing-Old-Photos-Back-to-Life"):
    subprocess.run(["git", "clone", "-q", "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life",
                    "/content/Bringing-Old-Photos-Back-to-Life"], check=True)
sys.path.insert(0, "/content")
!pip install -q lpips pytorch-fid pyiqa

## Code (identical to `src/`)

### `old_photo_model.py`

In [ ]:
%%writefile /content/old_photo_model.py
"""
OldPhotoModel: single-class wrapper around microsoft/Bringing-Old-Photos-Back-to-Life
(Global restoration stage). Finds the repo (Colab: /content/...), downloads the pretrained
weights if missing, builds VAE A, VAE B and the mapping network, loads their weights and
verifies that every checkpoint key matched.

    from old_photo_model import OldPhotoModel
    obj = OldPhotoModel(mode="quality", gpu=0)

    x   = obj.to_tensor(PIL_image)
    z_A = obj.encode_A(x)
    z_B = obj.map(z_A)
    y   = obj.decode_B(z_B)            # or obj.translate(x)
    obj.to_image(y).save("out.png")
    obj.restore(PIL_image).save("out.png")

Modes: "quality", "scratch" (needs mask), "scratch_hr" (needs mask, high-res).
Scratch modes: x, m = obj.prepare(image, mask); obj.map(z_A, m) / obj.translate(x, m).
"""
import contextlib
import glob
import os
import shutil
import sys
import tempfile
import urllib.request
import zipfile

import numpy as np
import torch
from PIL import Image, ImageFilter

_REPO_ZIP = "https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/archive/refs/heads/master.zip"
_CKPT_ZIP = ("https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/"
             "releases/download/v1.0/global_checkpoints.zip")
_DEFAULT_ROOT = "/content/Bringing-Old-Photos-Back-to-Life"

# (checkpoint sub-folder, network label) required per mode -> files named "<epoch>_net_<label>.pth"
_REQUIRED = {
    "quality":    [("VAE_A_quality", "G"), ("VAE_B_quality", "G"), ("mapping_quality", "mapping_net")],
    "scratch":    [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_scratch", "mapping_net")],
    "scratch_hr": [("VAE_A_quality", "G"), ("VAE_B_scratch", "G"), ("mapping_Patch_Attention", "mapping_net")],
}


def _download(url, dest):
    print(f"[OldPhotoModel] downloading {url}")
    with urllib.request.urlopen(url) as r, open(dest, "wb") as f:
        shutil.copyfileobj(r, f)


def _find_repo(hint=None):
    """Return repo root: hint, /content/<repo>, or any folder under /content containing Global/models/mapping_model.py."""
    marker = os.path.join("Global", "models", "mapping_model.py")
    cands = [hint, _DEFAULT_ROOT, _DEFAULT_ROOT + "-master"]
    cands += glob.glob("/content/*/") + glob.glob("/content/*/*/")
    for c in cands:
        if c and os.path.isfile(os.path.join(c, marker)):
            return os.path.abspath(c)
    return None


@contextlib.contextmanager
def _cpu_safe_torch_load():
    """The repo calls torch.load(path) without map_location, which fails on CPU-only machines."""
    orig = torch.load

    def patched(*args, **kwargs):
        kwargs.setdefault("map_location", "cpu")
        return orig(*args, **kwargs)

    torch.load = patched
    try:
        yield
    finally:
        torch.load = orig


class OldPhotoModel:

    def __init__(self, repo_root=None, mode="quality", gpu=0, auto_download=True, verify=True,
                 test_mode="Full"):
        assert mode in _REQUIRED, f"mode must be one of {list(_REQUIRED)}"
        assert test_mode in ("Full", "Scale"), "test_mode: 'Full' (keep size) or 'Scale' (short side -> 256)"
        self.mode = mode
        self.test_mode = test_mode  # only used in quality mode, like --test_mode in Global/test.py

        found = _find_repo(repo_root)
        if found is None:
            if not auto_download:
                raise FileNotFoundError("Repo not found under /content; pass repo_root=...")
            repo_root = os.path.abspath(repo_root or _DEFAULT_ROOT)
            self._ensure_repo(repo_root)  # zip download as fallback
        else:
            repo_root = found
        global_dir = os.path.join(repo_root, "Global")
        print(f"[OldPhotoModel] using repo at {repo_root}")

        if auto_download:
            self._ensure_weights(global_dir, mode)
        else:
            self._check_weights(global_dir, mode, raise_error=True)

        if gpu >= 0 and not torch.cuda.is_available():
            print("[OldPhotoModel] CUDA not available -> running on CPU")
            gpu = -1

        opt = self._build_opt(global_dir, mode, gpu)

        from models.mapping_model import Pix2PixHDModel_Mapping  # repo code

        prev = os.getcwd()
        os.chdir(global_dir)
        try:
            with _cpu_safe_torch_load():
                core = Pix2PixHDModel_Mapping()
                core.initialize(opt)  # builds netG_A, netG_B, mapping_net + loads weights
        finally:
            os.chdir(prev)

        self.vae_A = core.netG_A      # old-photo domain VAE
        self.vae_B = core.netG_B      # clean domain VAE
        self.mapping = core.mapping_net
        for net in (self.vae_A, self.vae_B, self.mapping):
            net.eval()
            for p in net.parameters():
                p.requires_grad = False

        self.uses_mask = bool(getattr(opt, "NL_use_mask", False))
        self._hr = bool(getattr(opt, "inference_optimize", False))
        self._mask_dilation = int(getattr(opt, "mask_dilation", 0) or 0)
        self.device = next(self.mapping.parameters()).device
        self._core = core
        self._opt = opt

        if verify:
            self._verify_weights(opt, mode)

    # ------------------------------------------------------------------ #
    # Core methods
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def encode_A(self, x):
        return self.vae_A.forward(x, flow="enc")

    @torch.no_grad()
    def encode_B(self, x):
        return self.vae_B.forward(x, flow="enc")

    @torch.no_grad()
    def map(self, z_A, mask=None):
        if not self.uses_mask:
            return self.mapping(z_A)
        if mask is None:
            raise ValueError("scratch mode: pass mask (1,1,H,W) tensor, same H/W as the input image "
                             "(use obj.prepare(image, mask)).")
        if self._hr:
            return self.mapping.inference_forward(z_A, mask)
        return self.mapping(z_A, mask)

    @torch.no_grad()
    def decode_A(self, z_A):
        return self.vae_A.forward(z_A, flow="dec")

    @torch.no_grad()
    def decode_B(self, z_B):
        return self.vae_B.forward(z_B, flow="dec")

    @torch.no_grad()
    def translate(self, x_A, mask=None):
        z_A = self.encode_A(x_A)
        z_B = self.map(z_A, mask)
        return self.decode_B(z_B)

    # ------------------------------------------------------------------ #
    # Helpers: PIL <-> tensor, and one-call restore
    # ------------------------------------------------------------------ #
    def _fit(self, img):
        """Same resizing as Global/test.py (BILINEAR, multiples of 4).
        Scratch modes never rescale in test.py, so only the multiple-of-4 rounding applies there."""
        w, h = img.size
        ow, oh = w, h
        if self.test_mode == "Scale" and not self.uses_mask:
            if w < h:
                ow, oh = 256, h / w * 256
            else:
                oh, ow = 256, w / h * 256
        size = (int(round(ow / 4) * 4), int(round(oh / 4) * 4))
        return img if size == (w, h) else img.resize(size, Image.BILINEAR)

    def to_tensor(self, image):
        """PIL -> (1,3,H,W) tensor in [-1,1]; H,W rounded to multiples of 4."""
        image = self._fit(image.convert("RGB"))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return ((x - 0.5) / 0.5).unsqueeze(0).to(self.device)

    def mask_to_tensor(self, mask, size):
        """PIL mask (white = scratch) -> (1,1,H,W) tensor, resized to size=(W,H)."""
        m = mask.convert("RGB").resize(size, Image.NEAREST)
        m = torch.from_numpy(np.asarray(m, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return m[:1].unsqueeze(0).to(self.device)

    def prepare(self, image, mask=None):
        """PIL (+ PIL mask in scratch modes) -> (x, mask_tensor or None), preprocessed like Global/test.py:
        size handled like test.py, mask dilated (HR mode), scratched pixels painted white."""
        img = self._fit(image.convert("RGB"))
        if not self.uses_mask:
            return self.to_tensor(img), None
        if mask is None:
            raise ValueError("scratch mode needs a mask image (white = scratch).")
        mk = mask.convert("RGB").resize(img.size, Image.NEAREST)
        for _ in range(self._mask_dilation):
            mk = mk.filter(ImageFilter.MaxFilter(3))
        m = np.asarray(mk, dtype=np.float32) / 255.0
        holed = np.asarray(img, dtype=np.float32) * (1.0 - m) + 255.0 * m
        img = Image.fromarray(holed.round().clip(0, 255).astype(np.uint8))
        return self.to_tensor(img), self.mask_to_tensor(mk, img.size)

    @staticmethod
    def to_image(x, normalize=True):
        """(1,3,H,W) tensor in [-1,1] -> PIL. normalize=True min-max stretches like
        save_image(..., normalize=True) in Global/test.py; False = plain clamp."""
        y = (x.detach().cpu().float() + 1.0) / 2.0
        if normalize:
            lo, hi = y.min(), y.max()
            y = (y - lo) / (hi - lo).clamp_min(1e-5)
        y = y.clamp(0, 1)[0]
        return Image.fromarray((y.permute(1, 2, 0).numpy() * 255).round().astype(np.uint8))

    def restore(self, image, mask=None, keep_size=True, normalize=True):
        """PIL in -> PIL out (full pipeline)."""
        x, m = self.prepare(image, mask)
        out = self.to_image(self.translate(x, m), normalize)
        if keep_size and out.size != image.size:
            out = out.resize(image.size, Image.BICUBIC)
        return out

    # ------------------------------------------------------------------ #
    # Setup: repo, weights, options
    # ------------------------------------------------------------------ #
    @staticmethod
    def _ensure_repo(repo_root):
        if os.path.isfile(os.path.join(repo_root, "Global", "models", "mapping_model.py")):
            return
        os.makedirs(os.path.dirname(repo_root), exist_ok=True)
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "repo.zip")
            _download(_REPO_ZIP, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(tmp)
            extracted = os.path.join(tmp, "Bringing-Old-Photos-Back-to-Life-master")
            if os.path.exists(repo_root):
                shutil.rmtree(repo_root)
            shutil.move(extracted, repo_root)

    def _ensure_weights(self, global_dir, mode):
        """Download the checkpoint zip (layout: checkpoints/restoration/<folder>/*.pth) and copy
        any missing or empty files for the needed folders."""
        if not self._missing(global_dir, mode):
            return
        target = os.path.join(global_dir, "checkpoints", "restoration")
        wanted = {f for f, _ in _REQUIRED[mode]}
        with tempfile.TemporaryDirectory() as tmp:
            zpath = os.path.join(tmp, "global_checkpoints.zip")
            _download(_CKPT_ZIP, zpath)
            extract_dir = os.path.join(tmp, "extracted")
            with zipfile.ZipFile(zpath) as z:
                z.extractall(extract_dir)
            src_root = os.path.join(extract_dir, "checkpoints", "restoration")
            for folder in wanted:
                os.makedirs(os.path.join(target, folder), exist_ok=True)
                for f in os.listdir(os.path.join(src_root, folder)):
                    src, dst = os.path.join(src_root, folder, f), os.path.join(target, folder, f)
                    if not os.path.isfile(dst) or os.path.getsize(dst) != os.path.getsize(src):
                        shutil.copy2(src, dst)
        self._check_weights(global_dir, mode, raise_error=True)

    @staticmethod
    def _missing(global_dir, mode):
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        missing = []
        for folder, label in _REQUIRED[mode]:
            if not glob.glob(os.path.join(ckpt, folder, f"*_net_{label}.pth")):
                missing.append(os.path.join(ckpt, folder, f"latest_net_{label}.pth"))
        return missing

    def _check_weights(self, global_dir, mode, raise_error):
        missing = self._missing(global_dir, mode)
        if missing and raise_error:
            raise FileNotFoundError("Missing checkpoints:\n  " + "\n  ".join(missing) +
                                    f"\nDownload {_CKPT_ZIP} and unzip it inside {global_dir}")
        return missing

    @staticmethod
    def _build_opt(global_dir, mode, gpu):
        if global_dir not in sys.path:
            sys.path.insert(0, global_dir)  # repo imports `options`, `models`, `util` from Global/
        from options.test_options import TestOptions

        flag = "--Quality_restore" if mode == "quality" else "--Scratch_and_Quality_restore"
        backup = sys.argv
        sys.argv = ["old_photo_model", flag, "--gpu_ids", str(gpu)]
        if mode == "scratch_hr":
            sys.argv.append("--HR")
        try:
            opt = TestOptions().parse(save=False)
        finally:
            sys.argv = backup

        # mirrors parameter_set() in Global/test.py
        ckpt = os.path.join(global_dir, "checkpoints", "restoration")
        opt.serial_batches = True
        opt.no_flip = True
        opt.label_nc = 0
        opt.n_downsample_global = 3
        opt.mc = 64
        opt.k_size = 4
        opt.start_r = 1
        opt.mapping_n_block = 6
        opt.map_mc = 512
        opt.no_instance = True
        opt.checkpoints_dir = ckpt

        if mode == "quality":
            opt.name = "mapping_quality"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_quality")
        else:
            opt.NL_res = True
            opt.use_SN = True
            opt.correlation_renormalize = True
            opt.NL_use_mask = True
            opt.NL_fusion_method = "combine"
            opt.non_local = "Setting_42"
            opt.name = "mapping_scratch"
            opt.load_pretrainA = os.path.join(ckpt, "VAE_A_quality")
            opt.load_pretrainB = os.path.join(ckpt, "VAE_B_scratch")
            if mode == "scratch_hr":
                opt.HR = True
                opt.mapping_exp = 1              # Mapping_Model_with_mask_2 (patch attention)
                opt.inference_optimize = True    # enables mapping.inference_forward
                opt.mask_dilation = 3
                opt.name = "mapping_Patch_Attention"
        return opt

    def _verify_weights(self, opt, mode):
        """Re-load each checkpoint with strict=True so a silent partial load can't go unnoticed."""
        ckpt = opt.checkpoints_dir
        vae_epoch = getattr(opt, "use_vae_which_epoch", "latest")
        map_epoch = getattr(opt, "which_epoch", "latest")
        jobs = [
            (self.vae_A, os.path.join(opt.load_pretrainA, f"{vae_epoch}_net_G.pth")),
            (self.vae_B, os.path.join(opt.load_pretrainB, f"{vae_epoch}_net_G.pth")),
            (self.mapping, os.path.join(ckpt, opt.name, f"{map_epoch}_net_mapping_net.pth")),
        ]
        for net, path in jobs:
            try:
                net.load_state_dict(torch.load(path, map_location="cpu"), strict=True)
            except Exception as e:  # noqa: BLE001
                raise RuntimeError(
                    f"Checkpoint {path} does not match the network built for mode '{mode}': {e}\n"
                    "The option flags in _build_opt() probably differ from Global/test.py."
                ) from e

### `degradation.py`

In [ ]:
%%writefile /content/degradation.py
"""
Port of the synthetic degradation pipeline of microsoft/Bringing-Old-Photos-Back-to-Life
(Global/data/online_dataset_for_old_photos.py): blur, Gaussian / speckle / salt-and-pepper noise,
downscaling and JPEG compression, applied in random order.
"""
import random
from io import BytesIO

import cv2
import numpy as np
from PIL import Image


def _pil_to_np(img):
    ar = np.array(img)
    ar = ar.transpose(2, 0, 1) if ar.ndim == 3 else ar[None, ...]
    return ar.astype(np.float32) / 255.0


def _np_to_pil(ar):
    ar = np.clip(ar * 255, 0, 255).astype(np.uint8)
    ar = ar[0] if ar.shape[0] == 1 else ar.transpose(1, 2, 0)
    return Image.fromarray(ar)


def synthesize_salt_pepper(image, amount, salt_vs_pepper):
    img = _pil_to_np(image)
    out = img.copy()
    flipped = np.random.choice([True, False], size=img.shape, p=[amount, 1 - amount])
    salted = np.random.choice([True, False], size=img.shape, p=[salt_vs_pepper, 1 - salt_vs_pepper])
    peppered = ~salted
    out[flipped & salted] = 1
    out[flipped & peppered] = 0.0
    return _np_to_pil(np.clip(out, 0, 1).astype(np.float32))


def synthesize_gaussian(image, std_l, std_r):
    img = _pil_to_np(image)
    std = random.uniform(std_l / 255.0, std_r / 255.0)
    noisy = img + np.random.normal(0, std, size=img.shape)
    return _np_to_pil(np.clip(noisy, 0, 1).astype(np.float32))


def synthesize_speckle(image, std_l, std_r):
    img = _pil_to_np(image)
    std = random.uniform(std_l / 255.0, std_r / 255.0)
    noisy = img + np.random.normal(0, std, size=img.shape) * img
    return _np_to_pil(np.clip(noisy, 0, 1).astype(np.float32))


def synthesize_low_resolution(img):
    w, h = img.size
    new_w = random.randint(int(w / 2), w)
    new_h = random.randint(int(h / 2), h)
    img = img.resize((new_w, new_h), Image.BICUBIC)
    method = Image.NEAREST if random.uniform(0, 1) < 0.5 else Image.BILINEAR
    return img.resize((w, h), method)


def convert_to_jpeg(img, quality):
    with BytesIO() as f:
        img.save(f, format="JPEG", quality=quality)
        f.seek(0)
        return Image.open(f).convert("RGB")


def blur_image_v2(img):
    x = np.array(img)
    kernel_size = random.choice([(3, 3), (5, 5), (7, 7)])
    std = random.uniform(1.0, 5.0)
    blur = cv2.GaussianBlur(x, kernel_size, std)
    return Image.fromarray(blur.astype(np.uint8))


def online_add_degradation_v2(img):
    """The repo's exact composite degradation used for training domain A / the mapping net."""
    task_id = np.random.permutation(4)
    for x in task_id:
        if x == 0 and random.uniform(0, 1) < 0.7:
            img = blur_image_v2(img)
        if x == 1 and random.uniform(0, 1) < 0.7:
            flag = random.choice([1, 2, 3])
            if flag == 1:
                img = synthesize_gaussian(img, 5, 50)
            elif flag == 2:
                img = synthesize_speckle(img, 5, 50)
            elif flag == 3:
                img = synthesize_salt_pepper(img, random.uniform(0, 0.01), random.uniform(0.3, 0.8))
        if x == 2 and random.uniform(0, 1) < 0.7:
            img = synthesize_low_resolution(img)
        if x == 3 and random.uniform(0, 1) < 0.7:
            img = convert_to_jpeg(img, random.randint(40, 100))
    return img


def irregular_hole_synthesize(img, mask):
    """Paint the masked (white) region white, exactly as the repo does before encoding domain A in scratch mode."""
    img_np = np.array(img).astype("uint8")
    mask_np = np.array(mask).astype("uint8") / 255
    img_new = img_np * (1 - mask_np) + mask_np * 255
    hole_img = Image.fromarray(img_new.astype("uint8")).convert("RGB")
    return hole_img, mask.convert("L")

### `online_dataset.py`

In [ ]:
%%writefile /content/online_dataset.py
"""
Training pairs generated on the fly: every __getitem__ draws a fresh clean image, a fresh random crop and a
fresh random degradation, as in the original BOPBTL training, so the model never sees the same degraded
instance twice.

Validation uses a fixed, in-memory set built once with a fixed seed from held-out source images (never used
for training), so validation numbers are comparable across runs.
"""
import glob
import math
import os
import random

import torch
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from degradation import online_add_degradation_v2

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".ppm")
FINE_SIZE = 256

_to_tensor_norm = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # -> [-1,1]
])


def list_clean_images(clean_dir, min_side=256):
    files = sorted(f for f in glob.glob(os.path.join(clean_dir, "*")) if f.lower().endswith(IMG_EXT))
    kept, corrupt = [], 0
    for f in files:
        try:
            with Image.open(f) as im:
                im.verify()          # catches truncated/corrupt files
            with Image.open(f) as im:  # verify() invalidates the handle, reopen to read size
                w, h = im.size
        except Exception:
            corrupt += 1
            continue
        if w >= min_side and h >= min_side:
            kept.append(f)
    print(f"{len(kept)}/{len(files)} images pass the >= {min_side}px filter "
          f"({corrupt} unreadable/corrupt files skipped)")
    return kept


def _crop_params(size, fine_size=FINE_SIZE):
    w, h = size
    x = random.randint(0, max(0, w - fine_size))
    y = random.randint(0, max(0, h - fine_size))
    flip = random.random() > 0.5
    return (x, y), flip


def _apply_crop_flip(img, pos, flip, fine_size=FINE_SIZE):
    ow, oh = img.size
    x1, y1 = pos
    if ow > fine_size or oh > fine_size:
        img = img.crop((x1, y1, x1 + fine_size, y1 + fine_size))
    if flip:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    return img


def make_online_pair(clean_path, grayscale_prob=0.1):
    """Fresh (A, B) tensor pair every call: new crop position, new flip, new
    degradation realization -- even for the same source image.

    grayscale_prob: fraction of pairs where BOTH A and B are desaturated
    (matches the repo's own training, which also keeps B grayscale when A is).
    Real target old photos are overwhelmingly B&W, so the default 0.1 (matching
    the original repo) under-represents that case for a mapping net trained
    from scratch without the repo's additional real-old-photo VAE_A domain
    adaptation -- raise this for training runs targeting B&W restoration."""
    B = Image.open(clean_path).convert("RGB")
    A = online_add_degradation_v2(B)
    if A.size != B.size:
        A = A.resize(B.size, Image.BILINEAR)

    if random.uniform(0, 1) < grayscale_prob:
        A = A.convert("L").convert("RGB")
        B = B.convert("L").convert("RGB")

    pos, flip = _crop_params(B.size)
    A = _apply_crop_flip(A, pos, flip)
    B = _apply_crop_flip(B, pos, flip)

    if A.size != (FINE_SIZE, FINE_SIZE):
        A = A.resize((FINE_SIZE, FINE_SIZE), Image.BICUBIC)
        B = B.resize((FINE_SIZE, FINE_SIZE), Image.BICUBIC)

    return _to_tensor_norm(A), _to_tensor_norm(B)


class OnlineQualityDataset(Dataset):
    """len() is nominal (controls how many samples count as "one epoch" for
    DataLoader bookkeeping / shuffling), but __getitem__ always draws a fresh
    random source image + fresh degradation, regardless of idx. This is the
    key difference from a dataset that reads a fixed PNG from disk per idx."""

    def __init__(self, clean_dir, length=20000, min_side=256, grayscale_prob=0.1):
        self.files = list_clean_images(clean_dir, min_side)
        assert self.files, f"no usable images (>= {min_side}px both sides) in {clean_dir}"
        self.length = length
        self.grayscale_prob = grayscale_prob

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        for _ in range(5):  # a handful of retries in case of a corrupt/transient-unreadable file
            f = random.choice(self.files)
            try:
                return make_online_pair(f, self.grayscale_prob)
            except Exception as e:
                print(f"[OnlineQualityDataset] skipping unreadable {f}: {e}")
        raise RuntimeError("too many consecutive unreadable files -- check your image folder")


def make_online_train_loader(clean_dir, batch_size=16, epoch_length=20000, num_workers=4, grayscale_prob=0.1):
    """Online training loader: every sample is freshly degraded. Use a
    multi-worker DataLoader since degradation (PIL/cv2 ops) is CPU-bound and
    would otherwise bottleneck the GPU."""
    ds = OnlineQualityDataset(clean_dir, length=epoch_length, grayscale_prob=grayscale_prob)
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0,
    )
    print(f"online train loader: {len(ds.files)} source images, "
          f"{epoch_length} samples/epoch, {len(loader)} batches/epoch")
    return loader


# --------------------------------------------------------------------------- #
# Train/val split at the SOURCE-IMAGE level, both drawn from the same folder.
# --------------------------------------------------------------------------- #
class FixedPairDataset(Dataset):
    """A small, pre-generated, in-memory set of (A, B) tensor pairs. Built
    once with a fixed seed so it's identical across runs/evals, but drawn
    from source images that were NEVER used for training."""

    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def make_online_dataloaders(clean_dir, batch_size=16, epoch_length=20000,
                             num_workers=2, val_fraction=0.05, val_samples=400, seed=0,
                             grayscale_prob=0.1):
    """Splits source images into train/val pools (no overlap), builds the
    online training loader from the train pool, and a FIXED val set (built
    once, with a fixed seed, same degradation pipeline) from the held-out
    val pool -- so val is comparable across runs AND actually measures
    generalization on the same visual domain the model is training on."""
    all_files = list_clean_images(clean_dir)
    rng = random.Random(seed)
    shuffled = all_files[:]
    rng.shuffle(shuffled)
    n_val_files = max(1, int(len(shuffled) * val_fraction))
    val_files, train_files = shuffled[:n_val_files], shuffled[n_val_files:]
    print(f"split: {len(train_files)} train source images, {len(val_files)} val source images (no overlap)")

    train_ds = OnlineQualityDataset.__new__(OnlineQualityDataset)  # bypass list_clean_images re-scan
    train_ds.files = train_files
    train_ds.length = epoch_length
    train_ds.grayscale_prob = grayscale_prob
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, drop_last=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0,
    )

    # build the fixed val set once, with a local seeded RNG so it's reproducible
    state = random.getstate()
    random.seed(seed + 1)
    pairs = []
    i = 0
    while len(pairs) < val_samples:
        f = val_files[i % len(val_files)]
        try:
            pairs.append(make_online_pair(f, grayscale_prob))
        except Exception as e:
            print(f"[val set] skipping unreadable {f}: {e}")
        i += 1
    random.setstate(state)

    val_loader = DataLoader(FixedPairDataset(pairs), batch_size=batch_size, shuffle=False, num_workers=0)
    print(f"online train loader: {len(train_files)} source images, {epoch_length} samples/epoch, "
          f"{len(train_loader)} batches/epoch | fixed val: {len(pairs)} samples from {len(val_files)} held-out images")
    return train_loader, val_loader

### `rf_model.py`

In [ ]:
%%writefile /content/rf_model.py
"""
Rectified flow model mapping z_A -> z_B in the frozen VAE latent space.

VelocityNet: v_theta(z_t, t | z_A), fully convolutional (works at any latent
resolution). FiLM-conditioned residual blocks, dilated convs for receptive
field, zero-initialized output so the flow starts as the identity.

RFMapping: wraps VelocityNet with the scalar latent normalization and the
Euler ODE integrator used both at train time (for validation samples) and at
inference time (obj.mapping replacement).

Bridge matching (sigma > 0): the straight path gets Brownian-bridge noise,
    z_t = (1 - t) z_A + t z_B + sigma * sqrt(t (1 - t)) * eps,
zero at both ends. The network output v defines the predicted endpoint
z_B_hat = z_t + (1 - t) v, trained against the true z_B, and sampling uses the
exact bridge posterior step
    z <- z + dt / (1 - t) * (z_B_hat - z) + sigma * sqrt(dt (1 - t') / (1 - t)) * eps.
A deterministic flow (sigma = 0) regresses to the AVERAGE of all clean images
consistent with z_A (soft texture, washed-out colour); with sigma > 0 the
sampler draws ONE plausible z_B instead. At sigma = 0 the sampler is exactly
the old Euler RF sampler, so RF checkpoints warm-start a bridge model directly.

direct_loss() (t = 0 ablation): only the point the 1-step sampler uses, z_t = z_A,
t = 0 -- plain direct regression z_A -> z_B with the same network, i.e. the
Microsoft-style mapping objective. No bridge noise, no t > 0.

traj_loss() trains the velocity at points the sampler actually reaches
(the model's own Euler trajectory), not only on the ideal straight line, so
the model learns to correct its own drift instead of accumulating it.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class TimeEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):  # t: (B,) in [0,1]
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = (t[:, None] * 1000.0) * freqs[None]
        return self.mlp(torch.cat([args.sin(), args.cos()], dim=1))


class FiLMResBlock(nn.Module):
    def __init__(self, channels, tdim, dilation=1, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.film = nn.Linear(tdim, 2 * channels)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        scale, shift = self.film(t_emb)[:, :, None, None].chunk(2, dim=1)
        h = self.norm2(h) * (1 + scale) + shift
        h = self.dropout(F.silu(h))
        return x + self.conv2(h)


class VelocityNet(nn.Module):
    """v(z_t, t | z_A). Input is concat(z_t, z_A) along channels."""

    def __init__(self, latent_ch, width=256, n_blocks=9, tdim=256, dropout=0.1):
        super().__init__()
        self.time_emb = TimeEmb(tdim)
        self.in_conv = nn.Conv2d(2 * latent_ch, width, 3, padding=1)
        dilations = (1, 2, 4)
        self.blocks = nn.ModuleList([
            FiLMResBlock(width, tdim, dilation=dilations[i % len(dilations)], dropout=dropout)
            for i in range(n_blocks)
        ])
        self.out_conv = nn.Conv2d(width, latent_ch, 3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, z_t, t, z_A):
        t_emb = self.time_emb(t)
        h = self.in_conv(torch.cat([z_t, z_A], dim=1))
        for block in self.blocks:
            h = block(h, t_emb)
        return self.out_conv(F.silu(h))


class RFMapping(nn.Module):
    """Rectified-flow replacement for the repo's mapping net.

    forward(z_A) integrates dz/dt = v(z, t | z_A) from t=0 (z_A) to t=1 (z_B)
    with `steps` Euler steps, matching the OldPhotoModel.mapping call signature
    closely enough to drop in via rf_mapping.install() / obj.mapping = rf.
    """

    def __init__(self, latent_ch, width=256, n_blocks=9, steps=8, sigma=0.0):
        super().__init__()
        self.latent_ch = latent_ch
        self.width = width
        self.n_blocks = n_blocks
        self.v = VelocityNet(latent_ch, width, n_blocks)
        self.steps = steps
        self.sigma = float(sigma)     # bridge noise (0 = deterministic RF); stored in checkpoint["arch"]
        self.sample_sigma = None      # inference override: None = use self.sigma, 0 = deterministic
        self.register_buffer("scale", torch.ones(()))  # latent std, set from training data

    def forward(self, z_A, mask=None, steps=None):
        return self.sample(z_A, steps)

    @staticmethod
    def _noise(z, generator=None):
        if generator is None:
            return torch.randn_like(z)
        return torch.randn(z.shape, generator=generator, device=z.device, dtype=z.dtype)

    @torch.no_grad()
    def sample(self, z_A, steps=None, sigma=None, generator=None):
        """n bridge-posterior steps from z_A. sigma: noise level (default:
        sample_sigma if set, else the trained sigma). sigma=0 is the old Euler
        RF sampler exactly. generator: optional torch.Generator for repeatable noise."""
        n = steps or self.steps
        if sigma is None:
            sigma = self.sigma if self.sample_sigma is None else self.sample_sigma
        a = z_A / self.scale
        z = a.clone()
        for i in range(n):
            t, t2 = i / n, (i + 1) / n
            z = z + self.v(z, torch.full((z.shape[0],), t, device=z.device), a) / n   # = z + dt/(1-t) (z_B_hat - z)
            if sigma > 0 and i < n - 1:
                z = z + sigma * math.sqrt((t2 - t) * (1 - t2) / (1 - t)) * self._noise(z, generator)
        return z * self.scale

    def rf_loss(self, z_A, z_B, time_sampler="logit_normal", return_endpoint=False):
        """Rectified-flow training loss: MSE between v_theta and the straight-line
        velocity (z_B - z_A), evaluated at a random point on the line.

        return_endpoint=True also returns z_B_hat = z_t + (1 - t) * v, the
        model's one-jump guess of z_B from that point (unscaled, differentiable),
        for image-space losses on decode_B(z_B_hat)."""
        a = z_A / self.scale
        b = z_B / self.scale
        bsz = a.shape[0]
        if time_sampler == "logit_normal":
            t = torch.sigmoid(torch.randn(bsz, device=a.device))
        else:
            t = torch.rand(bsz, device=a.device)
        tt = t[:, None, None, None]
        z_t = (1 - tt) * a + tt * b
        if self.sigma > 0:
            # bridge: noise on the path; target = drift towards z_B from the noisy point.
            # (b - z_t) / (1 - t) has noise / sqrt(1 - t), so the error is weighted by
            # (1 - t)^2 (= endpoint-space error), normalized to mean 1, as in traj_loss.
            t = t.clamp(max=0.999)
            tt = t[:, None, None, None]
            z_t = z_t + self.sigma * torch.sqrt(tt * (1 - tt)) * torch.randn_like(z_t)
            target = (b - z_t) / (1 - tt)
            pred = self.v(z_t, t, a)
            w = (1 - t) ** 2
            w = w / w.mean()
            loss = (w * ((pred - target) ** 2).flatten(1).mean(1)).mean()
        else:
            target = b - a
            pred = self.v(z_t, t, a)
            loss = F.mse_loss(pred, target)
        if not return_endpoint:
            return loss
        z_B_hat = (z_t + (1 - tt) * pred) * self.scale
        return loss, z_B_hat

    def direct_loss(self, z_A, z_B, return_endpoint=False):
        """t = 0 only: v(z_A, 0 | z_A) regresses z_B - z_A, so z_B_hat = z_A + v is
        exactly what sample(steps=1) returns. Same MSE (scaled latent) as rf_loss,
        which at t = 0 has the same target and weight."""
        a = z_A / self.scale
        b = z_B / self.scale
        t = torch.zeros(a.shape[0], device=a.device)
        pred = self.v(a, t, a)
        loss = F.mse_loss(pred, b - a)
        if not return_endpoint:
            return loss
        return loss, (a + pred) * self.scale

    @torch.no_grad()
    def rollout(self, a, k, n, sigma=0.0):
        """Per-sample partial sampling in scaled space: sample i takes the first
        k[i] of n[i] steps from a[i] (with bridge noise if sigma > 0, exactly as
        sample() does). a: (B,C,H,W) = z_A / scale; k, n: (B,) float tensors
        with k <= n - 1. Returns the reached points z."""
        z = a.clone()
        for i in range(int(k.max().item())):
            active = (i < k)[:, None, None, None]
            # finished samples (i >= k) are still in the batch: clamp their time below 1 so nothing
            # divides by zero, and select with where() -- 0 * nan would still be nan
            t = torch.minimum(torch.full_like(n, float(i)), n - 1) / n
            t2 = t + 1 / n
            step = self.v(z, t, a) / n[:, None, None, None]
            if sigma > 0:
                std = sigma * torch.sqrt((t2 - t) * (1 - t2) / (1 - t))
                step = step + std[:, None, None, None] * torch.randn_like(z)
            z = torch.where(active, z + step, z)
        return z

    def traj_loss(self, z_A, z_B, roller, step_choices=(4, 8, 16), return_endpoint=False, n_mode="sample"):
        """Like rf_loss, but at a point on the model's OWN trajectory: `roller`
        (normally the EMA copy, i.e. the weights used at inference) runs k of n
        Euler steps from z_A, with n drawn from step_choices and k uniform in
        [1, n-1] per sample. The target is the velocity that reaches the true
        z_B from there in the remaining time: (z_B - z) / (1 - t).

        The per-sample error is weighted by (1 - t)^2, i.e. measured where the
        jump lands ((z + (1-t) v) vs z_B) rather than in velocity, and the
        weights are normalized to mean 1. Without it, 1 / (1 - t) up to n
        amplifies drift late in the path (x256 in MSE at n=16) and those few
        samples dominate the loss.

        n_mode="batch" draws ONE sampler length n for the whole batch instead of
        one per sample. The rollout runs until the largest k in the batch, so
        with long samplers (e.g. 64) in the choices, per-sample n makes almost
        every batch pay ~n forward passes; per-batch n costs ~2.5x less on
        average and covers the same (n, k) distribution over training."""
        a = z_A / self.scale
        b = z_B / self.scale
        bsz = a.shape[0]
        choices = torch.tensor(step_choices, dtype=torch.float32, device=a.device)
        if n_mode == "batch":
            n = choices[torch.randint(len(step_choices), (1,), device=a.device)].expand(bsz)
        else:
            n = choices[torch.randint(len(step_choices), (bsz,), device=a.device)]
        k = torch.floor(torch.rand(bsz, device=a.device) * (n - 1)) + 1
        z = roller.rollout(a, k, n, self.sigma)
        t = k / n
        tt = t[:, None, None, None]
        target = (b - z) / (1 - tt)
        pred = self.v(z, t, a)
        w = (1 - t) ** 2
        w = w / w.mean()
        loss = (w * ((pred - target) ** 2).flatten(1).mean(1)).mean()
        if not return_endpoint:
            return loss
        z_B_hat = (z + (1 - tt) * pred) * self.scale
        return loss, z_B_hat

    def clone_for_ema(self):
        ema = RFMapping(self.latent_ch, self.width, self.n_blocks, self.steps, self.sigma)
        ema.load_state_dict(self.state_dict())
        for p in ema.parameters():
            p.requires_grad_(False)
        return ema

    @torch.no_grad()
    def ema_update(self, source, decay=0.999):
        for p_ema, p in zip(self.v.parameters(), source.v.parameters()):
            p_ema.mul_(decay).add_(p, alpha=1 - decay)
        self.scale.copy_(source.scale)  # scalar: just mirror, no EMA needed

### `direct_model.py`

In [ ]:
%%writefile /content/direct_model.py
"""
DirectMapping: the final z_A -> z_B mapping, a plain residual conv network in the
frozen VAE latent space (same mechanism as Microsoft's mapping network, ~3x smaller).

It is rf_model.VelocityNet with the time input removed. With t fixed to 0:
  - the time embedding is a constant vector, so each FiLM layer is a constant
    per-channel scale/shift right after norm2, which norm2's own affine absorbs;
  - the input concat(z_t, z_A) is concat(z_A, z_A), so the first conv's two weight
    halves merge into one conv on z_A.
So the function class is exactly that of the t = 0 model, and fold_rf() turns a
trained t = 0 RFMapping into a DirectMapping with identical outputs.

    z_B = scale * (a + net(a)),   a = z_A / scale

Residual, with a zero-initialized output conv, so training starts at the identity.
forward(z_A, mask=None) matches OldPhotoModel.mapping's call signature: obj.mapping = net.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResBlock(nn.Module):
    """FiLMResBlock without FiLM: GroupNorm -> SiLU -> dilated conv -> GroupNorm -> SiLU -> dropout -> conv, + skip."""

    def __init__(self, channels, dilation=1, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        h = self.conv1(F.silu(self.norm1(x)))
        h = self.dropout(F.silu(self.norm2(h)))
        return x + self.conv2(h)


class DirectMapping(nn.Module):
    def __init__(self, latent_ch, width=256, n_blocks=9, dropout=0.1):
        super().__init__()
        self.latent_ch, self.width, self.n_blocks = latent_ch, width, n_blocks
        self.in_conv = nn.Conv2d(latent_ch, width, 3, padding=1)
        dilations = (1, 2, 4)
        self.blocks = nn.ModuleList([ResBlock(width, dilations[i % len(dilations)], dropout)
                                     for i in range(n_blocks)])
        self.out_conv = nn.Conv2d(width, latent_ch, 3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)
        self.register_buffer("scale", torch.ones(()))  # latent std, set from training data

    def forward(self, z_A, mask=None):
        a = z_A / self.scale
        h = self.in_conv(a)
        for block in self.blocks:
            h = block(h)
        return (a + self.out_conv(F.silu(h))) * self.scale

    def arch(self):
        return {"type": "direct", "latent_ch": self.latent_ch, "width": self.width, "n_blocks": self.n_blocks}


@torch.no_grad()
def fold_rf(rf):
    """t = 0 RFMapping -> DirectMapping with identical outputs (both in eval mode)."""
    v, c = rf.v, rf.latent_ch
    d = DirectMapping(c, rf.width, rf.n_blocks).to(rf.scale.device)
    d.scale.copy_(rf.scale)
    w = v.in_conv.weight                                  # (width, 2c, 3, 3): halves for z_t and z_A
    d.in_conv.weight.copy_(w[:, :c] + w[:, c:])
    d.in_conv.bias.copy_(v.in_conv.bias)
    t_emb = v.time_emb(torch.zeros(1, device=w.device))   # constant at t = 0
    for bd, bv in zip(d.blocks, v.blocks):
        scale, shift = bv.film(t_emb)[0].chunk(2)
        for name in ("norm1", "conv1", "conv2"):
            getattr(bd, name).load_state_dict(getattr(bv, name).state_dict())
        # norm2(h) * (1 + scale) + shift = gn(h) * w (1 + scale) + (b (1 + scale) + shift)
        bd.norm2.weight.copy_(bv.norm2.weight * (1 + scale))
        bd.norm2.bias.copy_(bv.norm2.bias * (1 + scale) + shift)
    d.out_conv.load_state_dict(v.out_conv.state_dict())
    return d.eval()


def load_direct(path, device, use_ema=True):
    """DirectMapping from a train_direct checkpoint (or the exported weights file). Returns (net, step)."""
    ck = torch.load(path, map_location=device)
    arch = {k: v for k, v in ck["arch"].items() if k != "type"}
    net = DirectMapping(**arch).to(device)
    net.load_state_dict(ck["ema"] if use_ema else ck["net"])
    net.eval()
    print(f"loaded {path} | step {ck['step']} | {sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params")
    return net, ck["step"]

### `train_rf.py`

In [ ]:
%%writefile /content/train_rf.py
"""
Trains RFMapping (rf_model.py) to map z_A -> z_B, using (A, B) pixel pairs
from online_dataset.py and the frozen VAEs from OldPhotoModel.

- The VAEs are never trained; only rf.v's parameters get gradients.
- Loss = latent rectified-flow MSE + (optional) IMAGE-SPACE losses on the
  decoded endpoint prediction. Microsoft's own mapping net is trained with
  image-space losses (VGG / feature matching / GAN on decode_B output); a
  latent-only MSE leaves small latent errors that vae_B decodes into colour
  blotches and noise. The image terms are:
    pixel_weight      L1(decode_B(z_B_hat), B)
    chroma_weight     L1 on chroma (pixel minus its RGB mean) -- colour
                      fidelity; for gray targets it forces the output gray
    luma_weight       L1 on brightness (Y) -- tone + grain, gray and colour
    grad_weight       L1 on brightness gradients -- edges/detail; penalizes
                      both missing edges (blur) and extra ones (grain)
    perceptual_weight VGG16 feature L1 (can add grain if set too high)
    gan_weight /      LSGAN patch discriminator + its feature matching, the
    fm_weight         sharpness terms Microsoft's mapping training also uses
    gray_weight       multiplier on the per-sample terms for gray targets
                      (1.0 = gray and colour count equally)
    lowfreq_weight    L1 on 8x8 average-pooled images -- global tone /
                      brightness, which drifted in multi-step sampling
- Bridge matching (sigma > 0): see rf_model.py. sigma = 0 reproduces the
  deterministic RF exactly (same loss, same sampler). Validation samples use
  a fixed noise seed, so validation numbers are comparable across evals.
- Achievable target (target="vae"): the image losses compare the prediction
  with decode_B(encode_B(B)) -- the best image the frozen VAE B can produce
  -- instead of B itself, which even a perfect latent cannot reproduce (the
  VAE ceiling: dE00 ~5.3 vs ground truth). For gray targets the VAE
  reconstruction is replaced by its luminance, so gray stays exactly gray.
  The GAN / feature matching still use the real B as "real".
- Trajectory training (traj_prob > 0): on that fraction of steps the loss is
  taken at a point on the EMA model's own Euler trajectory instead of the
  ideal straight line (RFMapping.traj_loss), so the model learns to correct
  the drift that multi-step sampling accumulates (seen as a brightness bias
  that grew with the step count).
  z_B_hat is the one-jump endpoint estimate from a random point t on the
  straight path, so every t the sampler visits is supervised in image space,
  not only t=0.
- Validation encodes the held-out set once, then measures ODE-sampled latent
  MSE at a few step counts, plus (when image losses are on) decoded-image
  L1 / chroma / gradient error, reported separately for gray and colour
  targets. The best checkpoint is then picked by the average of the gray and
  colour scores, so neither can be sacrificed for the other.
- Checkpoints: latest.pt is saved every `save_every` steps, ATOMICALLY (written to
  a temp file, then renamed), with optimizer, scheduler, EMA and discriminator
  state, so a Colab disconnect never corrupts it; re-running train() with the
  same out_dir resumes from it. Optional snapshots every `snapshot_every` steps.
  With select_best=False / early_stop=False (fixed schedule, like Microsoft's
  training), no metric-based best.pt is written and the final checkpoint is used.
- t_mode="direct": every step uses RFMapping.direct_loss, i.e.
  only t = 0 and z_t = z_A -- the one point 1-step sampling evaluates. No bridge
  noise, no trajectory training; validation is 1-step only. t_mode is stored in
  checkpoint["arch"] and checked on resume.
- disc_type="multiscale": pix2pixHD-style discriminator as used by Microsoft's
  mapping training (Pix2PixHDModel_Mapping): num_D scales of an n_layers
  PatchGAN with instance norm, LSGAN, optionally conditional on the degraded
  input, feature matching weighted 4/(n_layers+1) * 1/num_D per layer;
  perceptual_type="vgg19": pix2pixHD VGGLoss (relu1_1..relu5_1, weights
  1/32..1). Microsoft weights both feature matching and VGG with lambda_feat=10.
"""
import argparse
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm

from rf_model import RFMapping


class VGGPerceptualLoss(nn.Module):
    """Feature-space loss: penalizes perceptual/structural difference, not just
    per-pixel color, so the network is pushed toward confident, realistic
    predictions instead of the pixelwise-MSE-optimal blurry average."""

    def __init__(self, device, layers=(3, 8, 15, 22)):
        super().__init__()
        vgg = tvm.vgg16(weights=tvm.VGG16_Weights.IMAGENET1K_V1).features.to(device).eval()
        for p in vgg.parameters():
            p.requires_grad_(False)
        self.vgg = vgg
        self.layers = set(layers)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1))

    def forward(self, pred, target):
        # pred/target are in [-1,1] (old_photo_model convention) -> back to [0,1] -> ImageNet norm
        pred_n = ((pred + 1) / 2 - self.mean) / self.std
        with torch.no_grad():  # target never needs gradients -- don't build/store its graph at all
            target_n = ((target + 1) / 2 - self.mean) / self.std
            y_feats = []
            y = target_n
            for i, layer in enumerate(self.vgg):
                y = layer(y)
                if i in self.layers:
                    y_feats.append(y)
                if i >= max(self.layers):
                    break

        loss = 0.0
        x = pred_n
        fi = 0
        for i, layer in enumerate(self.vgg):
            x = layer(x)
            if i in self.layers:
                loss = loss + F.l1_loss(x, y_feats[fi])
                fi += 1
            if i >= max(self.layers):
                break
        return loss


def chroma(x):
    """Per-pixel deviation from the RGB mean: exactly 0 for a gray image."""
    return x - x.mean(dim=1, keepdim=True)


def luma(x):
    """Brightness (BT.601 Y), (N,1,H,W): where grain and fine detail live."""
    return 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]


def per_sample_l1(x, y):
    """(N,) mean absolute error per sample."""
    return (x - y).abs().flatten(1).mean(1)


def grad_l1(x, y):
    """(N,) L1 between horizontal/vertical gradients of x and y: penalizes
    missing edges (blur) and extra edges (grain) alike."""
    dx = lambda t: t[..., :, 1:] - t[..., :, :-1]
    dy = lambda t: t[..., 1:, :] - t[..., :-1, :]
    return per_sample_l1(dx(x), dx(y)) + per_sample_l1(dy(x), dy(y))


def is_gray(x, tol=1e-3):
    """(N,) bool: the image has no colour."""
    return chroma(x).abs().flatten(1).amax(1) < tol


def lowfreq_l1(x, y, k=8):
    """(N,) L1 between k x k average-pooled images: only global tone / brightness."""
    return per_sample_l1(F.avg_pool2d(x, k), F.avg_pool2d(y, k))


def wmean(values, weights):
    return (values * weights).sum() / weights.sum()


class PatchDiscriminator(nn.Module):
    """Small PatchGAN with spectral norm. forward() -> (patch logits, features)."""

    def __init__(self, ch=64, n_layers=3):
        super().__init__()
        sn = nn.utils.spectral_norm
        layers, c_in = [], 3
        for i in range(n_layers):
            c_out = ch * 2 ** i
            layers.append(nn.Sequential(sn(nn.Conv2d(c_in, c_out, 4, 2, 1)), nn.LeakyReLU(0.2)))
            c_in = c_out
        layers.append(nn.Sequential(sn(nn.Conv2d(c_in, c_in, 4, 1, 1)), nn.LeakyReLU(0.2)))
        self.body = nn.ModuleList(layers)
        self.head = sn(nn.Conv2d(c_in, 1, 4, 1, 1))

    def forward(self, x):
        feats = []
        for layer in self.body:
            x = layer(x)
            feats.append(x)
        return self.head(x), feats


class NLayerDiscriminator(nn.Module):
    """pix2pixHD NLayerDiscriminator with intermediate features:
    forward() -> [feat_1, ..., feat_{n_layers+1}, logits]."""

    def __init__(self, input_nc=3, ndf=64, n_layers=3, norm="instance"):
        super().__init__()
        kw, padw = 4, 2
        norm_layer = (lambda c: nn.InstanceNorm2d(c, affine=False)) if norm == "instance" else (lambda c: nn.Identity())
        seq = [[nn.Conv2d(input_nc, ndf, kw, 2, padw), nn.LeakyReLU(0.2)]]
        nf = ndf
        for _ in range(1, n_layers):
            nf_prev, nf = nf, min(nf * 2, 512)
            seq.append([nn.Conv2d(nf_prev, nf, kw, 2, padw), norm_layer(nf), nn.LeakyReLU(0.2)])
        nf_prev, nf = nf, min(nf * 2, 512)
        seq.append([nn.Conv2d(nf_prev, nf, kw, 1, padw), norm_layer(nf), nn.LeakyReLU(0.2)])
        seq.append([nn.Conv2d(nf, 1, kw, 1, padw)])
        self.layers = nn.ModuleList(nn.Sequential(*m) for m in seq)

    def forward(self, x):
        out = []
        for layer in self.layers:
            x = layer(x)
            out.append(x)
        return out


class MultiscaleDiscriminator(nn.Module):
    """pix2pixHD MultiscaleDiscriminator: num_D NLayerDiscriminators on the
    image downsampled 1x, 2x, ... forward() -> list (one per scale) of feature lists."""

    def __init__(self, input_nc=3, ndf=64, n_layers=3, num_D=2, norm="instance"):
        super().__init__()
        self.num_D, self.n_layers = num_D, n_layers
        self.ds = nn.ModuleList(NLayerDiscriminator(input_nc, ndf, n_layers, norm) for _ in range(num_D))
        self.down = nn.AvgPool2d(3, stride=2, padding=1, count_include_pad=False)

    def forward(self, x):
        res = []
        for i, d in enumerate(self.ds):
            res.append(d(x))
            if i != self.num_D - 1:
                x = self.down(x)
        return res


def ms_gan_g_loss(preds_fake):
    """LSGAN generator loss, summed over scales (pix2pixHD GANLoss)."""
    return sum(((p[-1] - 1) ** 2).mean() for p in preds_fake)


def ms_gan_d_loss(preds_real, preds_fake):
    return 0.5 * (sum(((p[-1] - 1) ** 2).mean() for p in preds_real) + sum((p[-1] ** 2).mean() for p in preds_fake))


def ms_fm_loss(preds_fake, preds_real, n_layers, num_D):
    """pix2pixHD discriminator feature matching (all intermediate features, not the logits)."""
    feat_w, d_w = 4.0 / (n_layers + 1), 1.0 / num_D
    return sum(d_w * feat_w * F.l1_loss(pf[j], pr[j].detach())
               for pf, pr in zip(preds_fake, preds_real) for j in range(len(pf) - 1))


class VGG19Loss(nn.Module):
    """pix2pixHD VGGLoss: L1 between VGG19 relu1_1, relu2_1, relu3_1, relu4_1,
    relu5_1 features, weighted 1/32, 1/16, 1/8, 1/4, 1. Inputs in [-1,1] are
    mapped to ImageNet normalization first."""

    def __init__(self, device):
        super().__init__()
        vgg = tvm.vgg19(weights=tvm.VGG19_Weights.IMAGENET1K_V1).features.to(device).eval()
        for q in vgg.parameters():
            q.requires_grad_(False)
        cuts = (2, 7, 12, 21, 30)
        self.slices = nn.ModuleList(vgg[a:b] for a, b in zip((0,) + cuts[:-1], cuts))
        self.weights = (1 / 32, 1 / 16, 1 / 8, 1 / 4, 1.0)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1))

    def forward(self, pred, target):
        x = ((pred + 1) / 2 - self.mean) / self.std
        with torch.no_grad():
            y = ((target + 1) / 2 - self.mean) / self.std
            ys = []
            for sl in self.slices:
                y = sl(y)
                ys.append(y)
        loss = 0.0
        for sl, w, yf in zip(self.slices, self.weights, ys):
            x = sl(x)
            loss = loss + w * F.l1_loss(x, yf)
        return loss


# --------------------------------------------------------------------------- #
# Validation: precompute encoded val latents once, then ODE-sample at eval time
# --------------------------------------------------------------------------- #
@torch.no_grad()
def encode_loader(obj, loader, max_batches=None, keep_images=64):
    """Returns (zA, zB, B_images); B_images keeps the first `keep_images`
    targets (CPU) for the image-space validation metrics."""
    zA_all, zB_all, B_keep = [], [], []
    n_kept = 0
    for i, (A, B) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        if n_kept < keep_images:
            B_keep.append(B[:keep_images - n_kept])
            n_kept += B_keep[-1].shape[0]
        A, B = A.to(obj.device), B.to(obj.device)
        zA_all.append(obj.encode_A(A).cpu())
        zB_all.append(obj.encode_B(B).cpu())
    B_img = torch.cat(B_keep) if B_keep else None
    return torch.cat(zA_all), torch.cat(zB_all), B_img


@torch.no_grad()
def validate(rf, zA_val, zB_val, device, step_counts=(1, 4, 8), batch_size=16,
             obj=None, B_img=None, pix_steps=8):
    rf.eval()
    n = zA_val.shape[0]
    sums = {s: 0.0 for s in step_counts}
    gen = torch.Generator(device=device).manual_seed(1234)   # same bridge noise every eval
    for i in range(0, n, batch_size):
        a = zA_val[i:i + batch_size].to(device)
        b = zB_val[i:i + batch_size].to(device)
        for s in step_counts:
            pred = rf.sample(a, steps=s, generator=gen)
            sums[s] += F.mse_loss(pred, b, reduction="sum").item()
    numel = zA_val.numel()
    out = {s: v / numel for s, v in sums.items()}

    if obj is not None and B_img is not None:
        per = {"l1": [], "chroma": [], "grad": [], "bias": [], "gray": []}
        m = B_img.shape[0]
        for i in range(0, m, batch_size):
            j = min(i + batch_size, m)
            a = zA_val[i:j].to(device)
            tgt = B_img[i:j].to(device)
            img = obj.decode_B(rf.sample(a, steps=pix_steps, generator=gen))
            per["l1"].append(per_sample_l1(img, tgt).cpu())
            per["chroma"].append(per_sample_l1(chroma(img), chroma(tgt)).cpu())
            per["grad"].append(grad_l1(luma(img), luma(tgt)).cpu())
            per["bias"].append(((img - tgt) / 2).flatten(1).mean(1).cpu())  # brightness bias on a [0,1] scale
            per["gray"].append(is_gray(tgt).cpu())
        per = {k: torch.cat(v) for k, v in per.items()}
        scores = []
        for name, mask in (("gray", per["gray"]), ("color", ~per["gray"])):
            if mask.any():
                for k in ("l1", "chroma", "grad", "bias"):
                    out[f"{name}_{k}"] = per[k][mask].mean().item()
                scores.append(out[f"{name}_l1"] + out[f"{name}_chroma"] + out[f"{name}_grad"])
        out["img"] = sum(scores) / len(scores)  # gray and colour weighted equally
    return out


# --------------------------------------------------------------------------- #
# Checkpointing
# --------------------------------------------------------------------------- #
def save_checkpoint(path, rf, ema, opt, sched, step, best_val, patience_ct, extra=None):
    """Atomic: write to path.tmp, then rename, so an interrupted save (e.g. a Colab
    disconnect) never leaves a corrupt checkpoint behind."""
    tmp = path + ".tmp"
    torch.save({
        **(extra or {}),
        "rf": rf.state_dict(),
        "ema": ema.state_dict(),
        "opt": opt.state_dict(),
        "sched": sched.state_dict(),
        "step": step,
        "best_val": best_val,
        "patience_ct": patience_ct,
        "arch": {"latent_ch": rf.latent_ch, "width": rf.width, "n_blocks": rf.n_blocks, "sigma": rf.sigma,
                 "t_mode": getattr(rf, "t_mode", "bridge")},
    }, tmp)
    os.replace(tmp, path)


def load_checkpoint(path, device):
    return torch.load(path, map_location=device)


# --------------------------------------------------------------------------- #
# Training
# --------------------------------------------------------------------------- #
def train(
    obj,                      # OldPhotoModel instance (frozen VAEs)
    train_loader,
    val_loader,
    out_dir,
    width=256,
    n_blocks=9,
    lr=2e-4,
    weight_decay=1e-4,
    max_steps=20000,
    eval_every=500,
    ema_decay=0.999,
    patience=8,                # eval rounds without improvement before stopping
    val_max_batches=None,      # None = use the whole val_loader for encoding
    resume=True,
    init_from=None,            # path to a checkpoint whose EMA weights warm-start rf/ema;
                                # optimizer/scheduler/step/best_val/patience all start fresh.
                                # Ignored if `resume` finds an existing latest_path in out_dir.
    pixel_weight=0.0,          # L1 on decode_B(z_B_hat) vs B
    chroma_weight=0.0,         # L1 on chroma -- colour fidelity / anti-blotching
    luma_weight=0.0,           # L1 on brightness -- tone + grain
    grad_weight=0.0,           # L1 on brightness gradients -- detail without grain
    perceptual_weight=0.0,     # VGG16 feature L1 on decode_B(z_B_hat) vs B
    gan_weight=0.0,            # LSGAN patch-discriminator loss -- sharpness
    fm_weight=0.0,             # discriminator feature matching (needs gan_weight > 0)
    gan_start=0,               # step at which the GAN terms switch on
    d_lr=2e-4,                 # discriminator learning rate
    gray_weight=1.0,           # loss multiplier for gray targets (1.0 = equal to colour)
    lowfreq_weight=0.0,        # L1 on 8x8-pooled images -- global tone / brightness
    traj_prob=0.0,             # fraction of steps trained on the EMA model's own trajectory
    traj_steps=(4, 8, 16),     # sampler step counts the trajectory points are drawn from
    traj_n_mode="sample",      # "batch": one sampler length per batch (much cheaper with long samplers)
    target="clean",            # image-loss target: "clean" = B, "vae" = decode_B(encode_B(B)) (achievable)
    sigma=0.0,                 # bridge noise level (0 = deterministic RF)
    t_mode="bridge",           # "bridge": t ~ U[0,1] (+ trajectory); "direct": t = 0 only (ablation)
    disc_type="patch",         # "patch" (small PatchGAN) or "multiscale" (pix2pixHD, as Microsoft)
    num_D=2,                   # multiscale: number of scales
    n_layers_D=3,              # multiscale: layers per discriminator
    d_conditional=False,       # multiscale: discriminator also sees the degraded input (pix2pixHD cGAN)
    d_start=None,              # step at which the discriminator starts training (None = gan_start);
                                # lets a fresh discriminator warm up before it drives the generator
    perceptual_type="vgg16",   # "vgg16" or "vgg19" (pix2pixHD VGGLoss, as Microsoft)
    save_every=200,            # write latest.pt every N steps (crash-safe resume)
    snapshot_every=0,          # also keep step_XXXXXX.pt every N steps (0 = off)
    early_stop=True,           # False: fixed schedule, never stop early
    select_best=True,          # False: no metric-based best.pt; use the final checkpoint
    val_pix_steps=8,           # sampling steps for the image-space validation metrics
    pixel_every=1,             # only add the image-space terms every N steps, to control compute cost
    val_pix_images=128,        # val images kept for the image-space val metrics
):
    os.makedirs(out_dir, exist_ok=True)
    latest_path = os.path.join(out_dir, "latest.pt")
    best_path = os.path.join(out_dir, "best.pt")
    device = obj.device
    assert target in ("clean", "vae"), target
    assert traj_n_mode in ("sample", "batch"), traj_n_mode
    assert t_mode in ("bridge", "direct"), t_mode
    assert t_mode == "bridge" or traj_prob == 0, "t_mode='direct' trains only t = 0: set traj_prob=0"
    image_losses = any(w > 0 for w in (pixel_weight, chroma_weight, luma_weight, grad_weight,
                                       perceptual_weight, gan_weight, lowfreq_weight))

    # infer latent channel count from the loaded VAEs
    with torch.no_grad():
        latent_ch = obj.encode_A(torch.zeros(1, 3, 64, 64, device=device)).shape[1]

    rf = RFMapping(latent_ch, width, n_blocks, sigma=sigma).to(device)
    rf.t_mode = t_mode
    print(f"t_mode: {t_mode}" + (" (t = 0 only: direct regression z_A -> z_B)" if t_mode == "direct" else ""))

    if init_from and not (resume and os.path.exists(latest_path)):
        ck = load_checkpoint(init_from, device)
        rf.load_state_dict(ck["ema"])  # warm-start from the better (EMA) weights
        if ck["arch"].get("sigma", 0.0) != sigma:
            print(f"NOTE: init_from was trained with sigma={ck['arch'].get('sigma', 0.0)}, this run uses sigma={sigma}")
        print(f"warm-started weights from {init_from} (its step {ck['step']}, best_val {ck['best_val']:.5f}); "
              f"optimizer/scheduler/step/patience all reset")

    # encode the validation set once (VAEs frozen, so this doesn't change during training)
    print("encoding validation set...")
    zA_val, zB_val, B_val = encode_loader(obj, val_loader, val_max_batches,
                                          keep_images=val_pix_images if image_losses else 0)
    rf.scale.fill_(float(zB_val.float().std()))
    print(f"val set: {zA_val.shape[0]} samples | latent shape {tuple(zA_val.shape[1:])} | scale {float(rf.scale):.4f}")
    path_rms = float(((zB_val - zA_val).float() / rf.scale.cpu()).pow(2).mean().sqrt())
    print(f"bridge sigma {sigma} | RMS |z_B - z_A| (scaled) {path_rms:.3f} | "
          f"max path noise std sigma/2 = {sigma / 2:.3f} ({sigma / 2 / max(path_rms, 1e-6):.0%} of the path length)")

    ema = rf.clone_for_ema().to(device).eval()   # eval: no dropout in trajectory rollouts
    assert disc_type in ("patch", "multiscale") and perceptual_type in ("vgg16", "vgg19")
    d_start = gan_start if d_start is None else d_start
    perceptual = None
    if perceptual_weight > 0:
        perceptual = VGG19Loss(device) if perceptual_type == "vgg19" else VGGPerceptualLoss(device)
    opt = torch.optim.AdamW(rf.v.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max_steps)

    disc = opt_d = None
    if gan_weight > 0:
        if disc_type == "multiscale":
            disc = MultiscaleDiscriminator(6 if d_conditional else 3, 64, n_layers_D, num_D).to(device)
        else:
            disc = PatchDiscriminator().to(device)
        opt_d = torch.optim.Adam(disc.parameters(), lr=d_lr, betas=(0.5, 0.999))
        print(f"discriminator: {disc_type}" + (f" (num_D={num_D}, n_layers={n_layers_D}, "
              f"conditional={d_conditional})" if disc_type == "multiscale" else "")
              + f" | {sum(q.numel() for q in disc.parameters()) / 1e6:.2f}M params")

    def d_in(img, cond):
        return torch.cat([cond, img], 1) if (disc_type == "multiscale" and d_conditional) else img

    def extra_state():
        return ({"disc": disc.state_dict(), "opt_d": opt_d.state_dict(), "disc_type": disc_type,
                 "d_conditional": d_conditional} if disc is not None else None)

    start_step, best_val, patience_ct = 0, float("inf"), 0
    if resume and os.path.exists(latest_path):
        ck = load_checkpoint(latest_path, device)
        rf.load_state_dict(ck["rf"]); ema.load_state_dict(ck["ema"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        start_step, best_val, patience_ct = ck["step"], ck["best_val"], ck["patience_ct"]
        if ck["arch"].get("sigma", 0.0) != sigma:
            raise ValueError(f"{latest_path} was trained with sigma={ck['arch'].get('sigma', 0.0)} but sigma={sigma} "
                             f"was passed -- use the same sigma, or a new out_dir")
        if ck["arch"].get("t_mode", "bridge") != t_mode:
            raise ValueError(f"{latest_path} was trained with t_mode={ck['arch'].get('t_mode', 'bridge')} but "
                             f"t_mode={t_mode} was passed -- use a new out_dir")
        if disc is not None and "disc" in ck:
            if ck.get("disc_type", "patch") == disc_type and ck.get("d_conditional", False) == d_conditional:
                disc.load_state_dict(ck["disc"]); opt_d.load_state_dict(ck["opt_d"])
            else:
                print("NOTE: checkpoint has a different discriminator type -- discriminator starts fresh")
        print(f"RESUMED from {latest_path} at step {start_step}/{max_steps}")

    rf.train()
    terms = ("pix", "chroma", "luma", "grad", "low", "vgg")
    run = dict.fromkeys(("loss", "rf", *terms, "gan", "fm", "d"), 0.0)
    n_steps, n_img_steps, n_gan_steps, n_traj = 0, 0, 0, 0
    t0 = time.time()
    step = start_step
    train_iter = iter(train_loader)

    while step < max_steps:
        try:
            A, B = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            A, B = next(train_iter)

        A, B = A.to(device), B.to(device)
        with torch.no_grad():                     # VAEs stay frozen
            z_A = obj.encode_A(A)
            z_B = obj.encode_B(B)

        gan_on = False
        use_traj = traj_prob > 0 and torch.rand(()).item() < traj_prob
        n_traj += int(use_traj)
        if image_losses and step % pixel_every == 0:
            if use_traj:
                loss_rf, z_B_hat = rf.traj_loss(z_A, z_B, ema, traj_steps, return_endpoint=True,
                                                n_mode=traj_n_mode)
            elif t_mode == "direct":
                loss_rf, z_B_hat = rf.direct_loss(z_A, z_B, return_endpoint=True)
            else:
                # uniform t so the endpoint guess is supervised near t=0 too, where
                # few-step sampling starts (logit-normal rarely samples there)
                loss_rf, z_B_hat = rf.rf_loss(z_A, z_B, time_sampler="uniform", return_endpoint=True)
            # obj.decode_B is @torch.no_grad-wrapped for inference; call the VAE directly
            # so gradients flow from the image-space losses back into rf.v.
            pred_img = obj.vae_B.forward(z_B_hat, flow="dec")
            gray_b = is_gray(B)
            w = 1.0 + (gray_weight - 1.0) * gray_b.float()   # per-sample emphasis
            if target == "vae":
                with torch.no_grad():   # the best image VAE B can make from the true clean latent
                    tgt = obj.vae_B.forward(z_B, flow="dec")
                    if gray_b.any():    # gray targets stay exactly gray
                        tgt[gray_b] = luma(tgt[gray_b]).expand(-1, 3, -1, -1)
            else:
                tgt = B
            y_pred, y_tgt = luma(pred_img), luma(tgt)
            l = {
                "pix": wmean(per_sample_l1(pred_img, tgt), w),
                "chroma": wmean(per_sample_l1(chroma(pred_img), chroma(tgt)), w),
                "luma": wmean(per_sample_l1(y_pred, y_tgt), w),
                "grad": wmean(grad_l1(y_pred, y_tgt), w),
                "low": wmean(lowfreq_l1(pred_img, tgt), w),
                "vgg": perceptual(pred_img, tgt) if perceptual is not None else torch.zeros((), device=device),
            }
            loss = (loss_rf + pixel_weight * l["pix"] + chroma_weight * l["chroma"]
                    + luma_weight * l["luma"] + grad_weight * l["grad"] + lowfreq_weight * l["low"]
                    + perceptual_weight * l["vgg"])

            gan_on = disc is not None and step >= d_start          # discriminator trains this step
            g_adv = disc is not None and step >= gan_start         # adversarial terms drive the generator
            if g_adv:
                if disc_type == "multiscale":
                    p_fake = disc(d_in(pred_img, A))
                    with torch.no_grad():
                        p_real = disc(d_in(B, A))
                    l["gan"] = ms_gan_g_loss(p_fake)
                    l["fm"] = ms_fm_loss(p_fake, p_real, n_layers_D, num_D)
                else:
                    logits, f_fake = disc(pred_img)
                    with torch.no_grad():
                        _, f_real = disc(B)
                    l["gan"] = ((logits - 1) ** 2).mean()
                    l["fm"] = sum(F.l1_loss(a, b) for a, b in zip(f_fake, f_real)) / len(f_real)
                loss = loss + gan_weight * l["gan"] + fm_weight * l["fm"]
                n_gan_steps += 1

            for k, v in l.items():
                run[k] += v.item()
            n_img_steps += 1
        else:
            loss_rf = (rf.traj_loss(z_A, z_B, ema, traj_steps, n_mode=traj_n_mode) if use_traj
                       else rf.direct_loss(z_A, z_B) if t_mode == "direct" else rf.rf_loss(z_A, z_B))
            loss = loss_rf

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(rf.v.parameters(), 1.0)
        opt.step()
        sched.step()
        ema.ema_update(rf, decay=ema_decay)

        if gan_on:  # discriminator step: real B vs the (detached) prediction
            if disc_type == "multiscale":
                l_d = ms_gan_d_loss(disc(d_in(B, A)), disc(d_in(pred_img.detach(), A)))
            else:
                d_real, _ = disc(B)
                d_fake, _ = disc(pred_img.detach())
                l_d = 0.5 * (((d_real - 1) ** 2).mean() + (d_fake ** 2).mean())
            opt_d.zero_grad(set_to_none=True)
            l_d.backward()
            opt_d.step()
            run["d"] += l_d.item()

        run["loss"] += loss.item()
        run["rf"] += loss_rf.item()
        n_steps += 1
        step += 1

        if save_every and step % save_every == 0 and step % eval_every != 0:
            save_checkpoint(latest_path, rf, ema, opt, sched, step, best_val, patience_ct, extra_state())
        if snapshot_every and step % snapshot_every == 0:
            save_checkpoint(os.path.join(out_dir, f"step_{step:06d}.pt"), rf, ema, opt, sched, step,
                            best_val, patience_ct, extra_state())

        if step % eval_every == 0 or step == max_steps:
            msg = f"train_loss {run['loss'] / n_steps:.4f} (rf {run['rf'] / n_steps:.4f}"
            if n_img_steps:
                msg += "".join(f", {k} {run[k] / n_img_steps:.4f}" for k in terms)
            if n_gan_steps:
                msg += "".join(f", {k} {run[k] / n_gan_steps:.4f}" for k in ("gan", "fm", "d"))
            if traj_prob > 0:
                msg += f", traj {n_traj / n_steps:.0%} of steps"
            msg += ")"
            run = dict.fromkeys(run, 0.0)
            n_steps, n_img_steps, n_gan_steps, n_traj = 0, 0, 0, 0

            val = validate(ema, zA_val, zB_val, device, obj=obj, B_img=B_val, pix_steps=val_pix_steps,
                           step_counts=(1,) if t_mode == "direct" else (1, 4, 8))
            elapsed = time.time() - t0
            print(f"step {step}/{max_steps} | {msg}\n"
                  "    val_mse " + "  ".join(f"{s}-step {val[s]:.5f}" for s in (1, 4, 8) if s in val) +
                  f" | {elapsed:.0f}s | lr {sched.get_last_lr()[0]:.2e}")
            for name in ("gray", "color"):
                if f"{name}_l1" in val:
                    print(f"    val {name:5s}  l1 {val[name + '_l1']:.4f}  chroma {val[name + '_chroma']:.4f}"
                          f"  grad {val[name + '_grad']:.4f}  brightness bias {val[name + '_bias']:+.4f}"
                          f"  ({val_pix_steps}-step)")

            # best-checkpoint / early-stopping metric
            val_metric = val["img"] if "img" in val else val[8]

            if val_metric < best_val - 1e-6:
                best_val = val_metric
                patience_ct = 0
                if select_best:
                    save_checkpoint(best_path, rf, ema, opt, sched, step, best_val, patience_ct, extra_state())
                    print(f"  new best (val={best_val:.5f}), saved {best_path}")
            else:
                patience_ct += 1
                if select_best or early_stop:
                    print(f"  no improvement ({patience_ct}/{patience})")
            # saved after the best/patience update so a resume sees current values
            save_checkpoint(latest_path, rf, ema, opt, sched, step, best_val, patience_ct, extra_state())
            print(f"  saved {latest_path} (step {step})")
            if early_stop and patience_ct >= patience:
                print(f"early stopping at step {step} (best val={best_val:.5f})")
                break

            rf.train()
            t0 = time.time()

    save_checkpoint(latest_path, rf, ema, opt, sched, step, best_val, patience_ct, extra_state())
    if not select_best:
        final_path = os.path.join(out_dir, "final.pt")
        save_checkpoint(final_path, rf, ema, opt, sched, step, best_val, patience_ct, extra_state())
        print(f"training finished at step {step} -> {final_path} (fixed schedule, final checkpoint)")
    else:
        print(f"training finished at step {step}. best val={best_val:.5f} -> {best_path}")
    return rf, ema

### `train_direct.py`

In [ ]:
%%writefile /content/train_direct.py
"""
Trains the final DirectMapping (direct_model.py): z_A -> z_B with the frozen VAEs, using
Microsoft's loss types only:

    loss = MSE(z_B_hat / scale, z_B / scale)                 latent loss
         + vgg_weight * VGG19(decode_B(z_B_hat), B)          pix2pixHD VGGLoss
         + gan_weight * LSGAN(D(A, decode_B(z_B_hat)))       conditional multi-scale D   } from
         + fm_weight  * feature matching(D)                                              } gan_start

AdamW + cosine LR, gradient clipping 1.0, EMA 0.999 (the EMA weights are the model),
discriminator Adam(2e-4, betas 0.5/0.999) trained from d_start. Fixed schedule, no
metric-based checkpoint selection: final.pt is the model.

Crash-safe: latest.pt (all states) is written atomically at every evaluation (and every
save_every steps if > 0); re-running with the same out_dir resumes. history.csv keeps the
validation curve. At the end, final.pt and a small weights-only mapping_direct.pt (EMA).
"""
import os
import time

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm

from direct_model import DirectMapping


def chroma(x):
    """Per-pixel deviation from the RGB mean: exactly 0 for a gray image."""
    return x - x.mean(dim=1, keepdim=True)


def luma(x):
    """Brightness (BT.601 Y), (N,1,H,W)."""
    return 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]


def per_sample_l1(x, y):
    return (x - y).abs().flatten(1).mean(1)


def grad_l1(x, y):
    dx = lambda t: t[..., :, 1:] - t[..., :, :-1]
    dy = lambda t: t[..., 1:, :] - t[..., :-1, :]
    return per_sample_l1(dx(x), dx(y)) + per_sample_l1(dy(x), dy(y))


def is_gray(x, tol=1e-3):
    return chroma(x).abs().flatten(1).amax(1) < tol


# --- pix2pixHD discriminator, LSGAN, feature matching, VGG19 loss: same code as train_rf.py ---
class NLayerDiscriminator(nn.Module):
    """pix2pixHD NLayerDiscriminator with intermediate features:
    forward() -> [feat_1, ..., feat_{n_layers+1}, logits]."""

    def __init__(self, input_nc=3, ndf=64, n_layers=3, norm="instance"):
        super().__init__()
        kw, padw = 4, 2
        norm_layer = (lambda c: nn.InstanceNorm2d(c, affine=False)) if norm == "instance" else (lambda c: nn.Identity())
        seq = [[nn.Conv2d(input_nc, ndf, kw, 2, padw), nn.LeakyReLU(0.2)]]
        nf = ndf
        for _ in range(1, n_layers):
            nf_prev, nf = nf, min(nf * 2, 512)
            seq.append([nn.Conv2d(nf_prev, nf, kw, 2, padw), norm_layer(nf), nn.LeakyReLU(0.2)])
        nf_prev, nf = nf, min(nf * 2, 512)
        seq.append([nn.Conv2d(nf_prev, nf, kw, 1, padw), norm_layer(nf), nn.LeakyReLU(0.2)])
        seq.append([nn.Conv2d(nf, 1, kw, 1, padw)])
        self.layers = nn.ModuleList(nn.Sequential(*m) for m in seq)

    def forward(self, x):
        out = []
        for layer in self.layers:
            x = layer(x)
            out.append(x)
        return out


class MultiscaleDiscriminator(nn.Module):
    """pix2pixHD MultiscaleDiscriminator: num_D NLayerDiscriminators on the
    image downsampled 1x, 2x, ... forward() -> list (one per scale) of feature lists."""

    def __init__(self, input_nc=3, ndf=64, n_layers=3, num_D=2, norm="instance"):
        super().__init__()
        self.num_D, self.n_layers = num_D, n_layers
        self.ds = nn.ModuleList(NLayerDiscriminator(input_nc, ndf, n_layers, norm) for _ in range(num_D))
        self.down = nn.AvgPool2d(3, stride=2, padding=1, count_include_pad=False)

    def forward(self, x):
        res = []
        for i, d in enumerate(self.ds):
            res.append(d(x))
            if i != self.num_D - 1:
                x = self.down(x)
        return res


def ms_gan_g_loss(preds_fake):
    """LSGAN generator loss, summed over scales (pix2pixHD GANLoss)."""
    return sum(((p[-1] - 1) ** 2).mean() for p in preds_fake)


def ms_gan_d_loss(preds_real, preds_fake):
    return 0.5 * (sum(((p[-1] - 1) ** 2).mean() for p in preds_real) + sum((p[-1] ** 2).mean() for p in preds_fake))


def ms_fm_loss(preds_fake, preds_real, n_layers, num_D):
    """pix2pixHD discriminator feature matching (all intermediate features, not the logits)."""
    feat_w, d_w = 4.0 / (n_layers + 1), 1.0 / num_D
    return sum(d_w * feat_w * F.l1_loss(pf[j], pr[j].detach())
               for pf, pr in zip(preds_fake, preds_real) for j in range(len(pf) - 1))


class VGG19Loss(nn.Module):
    """pix2pixHD VGGLoss: L1 between VGG19 relu1_1, relu2_1, relu3_1, relu4_1,
    relu5_1 features, weighted 1/32, 1/16, 1/8, 1/4, 1. Inputs in [-1,1] are
    mapped to ImageNet normalization first."""

    def __init__(self, device):
        super().__init__()
        vgg = tvm.vgg19(weights=tvm.VGG19_Weights.IMAGENET1K_V1).features.to(device).eval()
        for q in vgg.parameters():
            q.requires_grad_(False)
        cuts = (2, 7, 12, 21, 30)
        self.slices = nn.ModuleList(vgg[a:b] for a, b in zip((0,) + cuts[:-1], cuts))
        self.weights = (1 / 32, 1 / 16, 1 / 8, 1 / 4, 1.0)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1))

    def forward(self, pred, target):
        x = ((pred + 1) / 2 - self.mean) / self.std
        with torch.no_grad():
            y = ((target + 1) / 2 - self.mean) / self.std
            ys = []
            for sl in self.slices:
                y = sl(y)
                ys.append(y)
        loss = 0.0
        for sl, w, yf in zip(self.slices, self.weights, ys):
            x = sl(x)
            loss = loss + w * F.l1_loss(x, yf)
        return loss


@torch.no_grad()
def encode_loader(obj, loader, keep_images=128):
    """Encodes the (fixed) validation set once: (zA, zB, first keep_images clean targets)."""
    zA, zB, keep, n_kept = [], [], [], 0
    for A, B in loader:
        if n_kept < keep_images:
            keep.append(B[:keep_images - n_kept])
            n_kept += keep[-1].shape[0]
        zA.append(obj.encode_A(A.to(obj.device)).cpu())
        zB.append(obj.encode_B(B.to(obj.device)).cpu())
    return torch.cat(zA), torch.cat(zB), torch.cat(keep)


@torch.no_grad()
def validate(net, obj, zA_val, zB_val, B_img, batch_size=16):
    """Latent MSE over the whole val set + image-space L1 / chroma / gradient / brightness bias,
    separately for gray and colour targets ("img" = their average, gray and colour equally weighted)."""
    device = obj.device
    net.eval()
    sse = 0.0
    for i in range(0, len(zA_val), batch_size):
        sse += F.mse_loss(net(zA_val[i:i + batch_size].to(device)), zB_val[i:i + batch_size].to(device),
                          reduction="sum").item()
    out = {"latent_mse": sse / zA_val.numel()}
    per = {"l1": [], "chroma": [], "grad": [], "bias": [], "gray": []}
    for i in range(0, len(B_img), batch_size):
        j = min(i + batch_size, len(B_img))
        img = obj.decode_B(net(zA_val[i:j].to(device)))
        tgt = B_img[i:j].to(device)
        per["l1"].append(per_sample_l1(img, tgt).cpu())
        per["chroma"].append(per_sample_l1(chroma(img), chroma(tgt)).cpu())
        per["grad"].append(grad_l1(luma(img), luma(tgt)).cpu())
        per["bias"].append(((img - tgt) / 2).flatten(1).mean(1).cpu())
        per["gray"].append(is_gray(tgt).cpu())
    per = {k: torch.cat(v) for k, v in per.items()}
    scores = []
    for name, mask in (("gray", per["gray"]), ("color", ~per["gray"])):
        if mask.any():
            for k in ("l1", "chroma", "grad", "bias"):
                out[f"{name}_{k}"] = per[k][mask].mean().item()
            scores.append(out[f"{name}_l1"] + out[f"{name}_chroma"] + out[f"{name}_grad"])
    out["img"] = sum(scores) / len(scores)
    return out


def _atomic_save(obj_, path):
    tmp = path + ".tmp"
    torch.save(obj_, tmp)
    os.replace(tmp, path)


def train_direct(
    obj, train_loader, val_loader, out_dir,
    width=256, n_blocks=9, dropout=0.1,
    lr=2e-4, weight_decay=1e-4, max_steps=20000, eval_every=500, ema_decay=0.999,
    vgg_weight=10.0, gan_weight=1.0, fm_weight=10.0,
    gan_start=2000, d_start=0, d_lr=2e-4,
    num_D=2, n_layers_D=3, d_conditional=True,
    val_pix_images=128, save_every=0, resume=True,
):
    os.makedirs(out_dir, exist_ok=True)
    latest_path, hist_path = f"{out_dir}/latest.pt", f"{out_dir}/history.csv"
    device = obj.device
    with torch.no_grad():
        latent_ch = obj.encode_A(torch.zeros(1, 3, 64, 64, device=device)).shape[1]
    net = DirectMapping(latent_ch, width, n_blocks, dropout).to(device)
    print(f"DirectMapping: {sum(p.numel() for p in net.parameters()) / 1e6:.2f}M params")

    print("encoding validation set...")
    zA_val, zB_val, B_val = encode_loader(obj, val_loader, val_pix_images)
    net.scale.fill_(float(zB_val.float().std()))
    print(f"val set: {zA_val.shape[0]} samples | latent {tuple(zA_val.shape[1:])} | scale {float(net.scale):.4f}")

    ema = DirectMapping(latent_ch, width, n_blocks, dropout).to(device).eval()
    ema.load_state_dict(net.state_dict())
    for p in ema.parameters():
        p.requires_grad_(False)
    vgg = VGG19Loss(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max_steps)
    disc = MultiscaleDiscriminator(6 if d_conditional else 3, 64, n_layers_D, num_D).to(device)
    opt_d = torch.optim.Adam(disc.parameters(), lr=d_lr, betas=(0.5, 0.999))
    d_in = (lambda img, cond: torch.cat([cond, img], 1)) if d_conditional else (lambda img, cond: img)
    print(f"discriminator: multiscale (num_D={num_D}, n_layers={n_layers_D}, conditional={d_conditional}) | "
          f"{sum(q.numel() for q in disc.parameters()) / 1e6:.2f}M params")

    def save(path, step):
        _atomic_save({"net": net.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(),
                      "sched": sched.state_dict(), "disc": disc.state_dict(), "opt_d": opt_d.state_dict(),
                      "step": step, "arch": net.arch()}, path)

    step, history = 0, []
    if resume and os.path.exists(latest_path):
        ck = torch.load(latest_path, map_location=device)
        if ck["arch"] != net.arch():
            raise ValueError(f"{latest_path} has arch {ck['arch']}, this run builds {net.arch()}: use a new out_dir")
        net.load_state_dict(ck["net"]); ema.load_state_dict(ck["ema"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        disc.load_state_dict(ck["disc"]); opt_d.load_state_dict(ck["opt_d"])
        step = ck["step"]
        if os.path.exists(hist_path):
            history = [r for r in pd.read_csv(hist_path).to_dict("records") if r["step"] <= step]
        print(f"RESUMED from {latest_path} at step {step}/{max_steps}")

    net.train()
    run = dict.fromkeys(("loss", "latent", "vgg", "gan", "fm", "d"), 0.0)
    n_steps, n_gan = 0, 0
    t0 = time.time()
    train_iter = iter(train_loader)
    while step < max_steps:
        try:
            A, B = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            A, B = next(train_iter)
        A, B = A.to(device), B.to(device)
        with torch.no_grad():                       # VAEs stay frozen
            z_A, z_B = obj.encode_A(A), obj.encode_B(B)

        z_hat = net(z_A)
        l_lat = F.mse_loss(z_hat / net.scale, z_B / net.scale)
        pred_img = obj.vae_B.forward(z_hat, flow="dec")   # not obj.decode_B: that one is no_grad
        l_vgg = vgg(pred_img, B)
        loss = l_lat + vgg_weight * l_vgg
        if step >= gan_start:
            p_fake = disc(d_in(pred_img, A))
            with torch.no_grad():
                p_real = disc(d_in(B, A))
            l_gan, l_fm = ms_gan_g_loss(p_fake), ms_fm_loss(p_fake, p_real, n_layers_D, num_D)
            loss = loss + gan_weight * l_gan + fm_weight * l_fm
            run["gan"] += l_gan.item()
            run["fm"] += l_fm.item()
            n_gan += 1

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        sched.step()
        with torch.no_grad():
            for p_e, p in zip(ema.parameters(), net.parameters()):
                p_e.mul_(ema_decay).add_(p, alpha=1 - ema_decay)
            ema.scale.copy_(net.scale)

        if step >= d_start:                         # discriminator: real B vs the detached prediction
            l_d = ms_gan_d_loss(disc(d_in(B, A)), disc(d_in(pred_img.detach(), A)))
            opt_d.zero_grad(set_to_none=True)
            l_d.backward()
            opt_d.step()
            run["d"] += l_d.item()

        run["loss"] += loss.item()
        run["latent"] += l_lat.item()
        run["vgg"] += l_vgg.item()
        n_steps += 1
        step += 1

        if save_every and step % save_every == 0 and step % eval_every != 0:
            save(latest_path, step)
        if step % eval_every == 0 or step == max_steps:
            tr = {k: run[k] / (n_gan if k in ("gan", "fm") and n_gan else n_steps) for k in run}
            val = validate(ema, obj, zA_val, zB_val, B_val)
            net.train()
            history.append({"step": step, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in val.items()}})
            hist_tmp = hist_path + ".tmp"
            pd.DataFrame(history).to_csv(hist_tmp, index=False)
            os.replace(hist_tmp, hist_path)
            save(latest_path, step)
            print(f"step {step}/{max_steps} | loss {tr['loss']:.4f} (latent {tr['latent']:.4f}, vgg {tr['vgg']:.4f}, "
                  f"gan {tr['gan']:.4f}, fm {tr['fm']:.4f}, d {tr['d']:.4f}) | {time.time() - t0:.0f}s | "
                  f"lr {sched.get_last_lr()[0]:.2e}\n"
                  f"    val latent_mse {val['latent_mse']:.5f} | img {val['img']:.4f}"
                  + "".join(f" | {n} l1 {val[n + '_l1']:.4f} bias {val[n + '_bias']:+.4f}"
                            for n in ("gray", "color") if f"{n}_l1" in val)
                  + f"\n    saved {latest_path} (step {step})")
            run = dict.fromkeys(run, 0.0)
            n_steps, n_gan = 0, 0
            t0 = time.time()

    save(latest_path, step)
    save(f"{out_dir}/final.pt", step)
    _atomic_save({"ema": ema.state_dict(), "arch": ema.arch(), "step": step}, f"{out_dir}/mapping_direct.pt")
    print(f"training finished at step {step} -> {out_dir}/final.pt (+ weights-only mapping_direct.pt)")
    return net, ema

## Train

In [ ]:
ROOT = "/content/drive/MyDrive/lightweight_old_photos"      # project folder on Google Drive
MODEL = "direct"                  # "direct" (the final model) or "bridge_flow" (ablation)
SEED = 0
CLEAN_DIR = f"{ROOT}/data/voc_clean"
OUT_DIR = f"{ROOT}/checkpoints/{MODEL}" + (f"_seed{SEED}" if SEED else "")

import random, shutil
import numpy as np
import torch
from old_photo_model import OldPhotoModel
from online_dataset import make_online_dataloaders

free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
assert free_gb >= 2.0, f"only {free_gb:.1f} GB free on Google Drive: checkpoints would be lost"
obj = OldPhotoModel(mode="quality", gpu=0)
train_loader, val_loader = make_online_dataloaders(clean_dir=CLEAN_DIR, batch_size=8, epoch_length=20000,
                                                   num_workers=2, val_fraction=0.05, val_samples=400,
                                                   grayscale_prob=0.5)
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
common = dict(width=256, n_blocks=9, lr=2e-4, weight_decay=1e-4, max_steps=20000, eval_every=500,
              gan_start=2000, d_start=0, d_lr=2e-4, num_D=2, n_layers_D=3, d_conditional=True)
if MODEL == "direct":
    from train_direct import train_direct
    train_direct(obj, train_loader, val_loader, OUT_DIR, vgg_weight=10.0, gan_weight=1.0, fm_weight=10.0, **common)
else:
    from train_rf import train
    train(obj, train_loader, val_loader, OUT_DIR, sigma=0.5, t_mode="bridge", traj_prob=0.5, traj_steps=(4, 8, 16),
          disc_type="multiscale", gan_weight=1.0, fm_weight=10.0, perceptual_type="vgg19", perceptual_weight=10.0,
          val_pix_steps=1, pixel_every=1, early_stop=False, select_best=False, save_every=0, snapshot_every=0,
          **common)